# Кандидатогенерация

## Настройки

In [1]:
%pip -q install "huggingface_hub>=0.30" "pyarrow>=16" "scipy>=1.11" tqdm

In [2]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from scipy import sparse
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download
from google.colab import drive

drive.mount('/content/drive')
OUT = Path('/content/drive/MyDrive/TLab_Yambda/retrieval_gts_v2')
OUT.mkdir(parents=True, exist_ok=True)

SEED = 666
N_USERS = 10000
N_CANDIDATES = 600
N_ITEMKNN = 450
RECENT_POSITIVES = 80
PROFILE_HALFLIFE_DAYS = 7
POP_HALFLIFE_DAYS = 14
DAY = 24 * 60 * 60
GAP = 30 * 60
H = DAY
LAST_TIMESTAMP = 26000000
TEST_START = LAST_TIMESTAMP - DAY
REPO = 'yandex/yambda'
SOURCE = 'flat/50m/multi_event.parquet'

import subprocess, sys
with (OUT/'requirements-lock.txt').open('w') as lockfile:
    subprocess.run([sys.executable, '-m', 'pip', 'freeze'], stdout=lockfile, check=True)


Mounted at /content/drive


## Пользователи

In [3]:
source_path = hf_hub_download(repo_id=REPO, repo_type='dataset', filename=SOURCE)
pq_file = pq.ParquetFile(source_path)

all_users = set()
for batch in tqdm(pq_file.iter_batches(batch_size=500_000, columns=['uid']), desc='UIDs'):
    all_users.update(batch.column(0).to_numpy())

rng = np.random.default_rng(SEED)
selected_users = np.sort(rng.choice(np.array(sorted(all_users)), N_USERS, replace=False)).astype('uint32')
pd.DataFrame({'uid': selected_users}).to_csv(OUT / 'selected_users.csv', index=False)
print('Выбрано пользователей:', len(selected_users))


flat/50m/multi_event.parquet: reconstructing file:   0%|          |  0.00B /  384MB            

flat/50m/multi_event.parquet: downloading bytes:           |  0.00B            

UIDs: 0it [00:00, ?it/s]

Выбрано пользователей: 10000


In [4]:
parts = []
cols = ['uid', 'item_id', 'timestamp', 'event_type', 'played_ratio_pct',
        'is_organic', 'track_length_seconds']
for batch in tqdm(pq_file.iter_batches(batch_size=500000, columns=cols), desc='Events'):
    part = batch.to_pandas()
    part = part[part.uid.isin(selected_users)]
    if len(part):
        parts.append(part)

events = pd.concat(parts, ignore_index=True)
events = events.sort_values(['timestamp', 'uid'], kind='stable').reset_index(drop=True)
events['positive'] = events.event_type.eq('listen') & events.played_ratio_pct.ge(50)
events.to_parquet(OUT / 'events_sample.parquet', index=False)
print('События:', f'{len(events):,}', '· Listen+:', f'{events.positive.sum():,}')
display(events.event_type.value_counts().to_frame('events'))


Events: 0it [00:00, ?it/s]

События: 47,790,449 · Listen+: 29,439,278


,events
event_type,
listen,46467212
like,881456
unlike,312972
dislike,107776
undislike,21033


## Временные срезы

In [5]:
VALID_START = TEST_START - GAP - DAY
TRAIN_START = VALID_START - GAP - DAY
starts = [TRAIN_START, VALID_START, TEST_START]
snapshots = pd.DataFrame({
    'snapshot_id': [0, 1, 2],
    'split': ['train', 'valid', 'test'],
    'cutoff': starts,
    'history_end': [t - GAP for t in starts],
    'end': [t + DAY for t in starts],
})
snapshots.to_csv(OUT / 'snapshots.csv', index=False)
display(snapshots)


,snapshot_id,split,cutoff,history_end,end
0,0,train,25737200,25735400,25823600
1,1,valid,25825400,25823600,25911800
2,2,test,25913600,25911800,26000000


## Retrieval

In [6]:
def build_retriever(past, cutoff, user_ids):

    pos = past.loc[past.positive, ['uid', 'item_id', 'timestamp']]
    unique = pos[['uid', 'item_id']].drop_duplicates()
    items, col = np.unique(unique.item_id.to_numpy(), return_inverse=True)
    row = pd.Index(user_ids).get_indexer(unique.uid)

    X = sparse.coo_matrix((np.ones(len(unique), dtype='float32'), (row, col)),
                          shape=(len(user_ids), len(items))).tocsr()
    norm = np.sqrt(np.asarray(X.sum(axis=0)).ravel())
    X = X.multiply(1 / np.maximum(norm, 1)).tocsr()

    recent = pos.sort_values('timestamp').groupby('uid', sort=False).tail(RECENT_POSITIVES)
    r = pd.Index(user_ids).get_indexer(recent.uid)
    c = pd.Index(items).get_indexer(recent.item_id)
    age = np.int64(cutoff) - recent.timestamp.to_numpy(dtype=np.int64)
    decay = np.exp2(-age / (DAY * PROFILE_HALFLIFE_DAYS))
    R = sparse.coo_matrix((decay.astype('float32'), (r, c)), shape=X.shape).tocsr()

    age_pop = np.int64(cutoff) - pos.timestamp.to_numpy(dtype=np.int64)
    weights = np.exp2(-age_pop / (DAY * POP_HALFLIFE_DAYS))
    popularity = pd.DataFrame({'item_id': pos.item_id.to_numpy(), 'w': weights})
    pop_scores = popularity.groupby('item_id').w.sum().sort_values(ascending=False)
    pop_items = pop_scores.index.to_numpy(dtype='uint32')
    return items, X, R, pop_items, pop_scores


def recommend(uid_index, items, X, R, pop_items, pop_scores):
    profile = R.getrow(uid_index)
    scores = ((profile @ X.T) @ X - profile).tocsr()
    keep = scores.data > 0
    col = scores.indices[keep]
    values = scores.data[keep]
    order = np.argsort(-values, kind='stable')[:N_ITEMKNN]

    chosen = [int(items[c]) for c in col[order]]
    knn_scores = {int(items[col[j]]): float(values[j]) for j in order}
    source = ['itemknn'] * len(chosen)
    seen = set(chosen)
    for item in pop_items:
        item = int(item)
        if item not in seen:
            chosen.append(item)
            source.append('decaypop')
            seen.add(item)
        if len(chosen) == N_CANDIDATES:
            break

    return pd.DataFrame({
        'item_id': np.array(chosen, dtype='uint32'),
        'retrieval_rank': np.arange(1, len(chosen) + 1, dtype='int16'),
        'retrieval_source': source,
        'itemknn_score': [knn_scores.get(x, 0.) for x in chosen],
        'decaypop_score': [float(pop_scores.get(x, 0.)) for x in chosen],
    })


## Кандидаты

In [7]:
tables, retrieval_stats = [], []
for snap in snapshots.itertuples(index=False):
    t = int(snap.cutoff)
    past = events[events.timestamp < t - GAP]
    future = events[(events.timestamp >= t) & (events.timestamp < t + H) & events.positive]
    targets = future.groupby('uid').item_id.agg(lambda a: set(a))
    items, X, R, pop_items, pop_scores = build_retriever(past, t, selected_users)

    hit_count = true_count = knn_hits = groups_positive = 0
    for uindex, uid in enumerate(tqdm(selected_users, desc=f'{snap.split} · {snap.snapshot_id}', leave=False)):
        group = recommend(uindex, items, X, R, pop_items, pop_scores)
        target = targets.get(uid, set())
        group['uid'] = int(uid)
        group['snapshot_id'] = int(snap.snapshot_id)
        group['cutoff'] = t
        group['split'] = snap.split
        group['label'] = group.item_id.isin(target).astype('int8')
        group['group_id'] = int(snap.snapshot_id) * len(selected_users) + uindex
        tables.append(group)

        hit_count += int(group.label.sum())
        true_count += len(target)
        groups_positive += bool(group.label.any())
        knn_hits += int(group.loc[group.retrieval_source.eq('itemknn'), 'label'].sum())

    retrieval_stats.append({
        'snapshot_id': int(snap.snapshot_id), 'split': snap.split,
        'candidate_recall': hit_count / true_count if true_count else np.nan,
        'itemknn_recall': knn_hits / true_count if true_count else np.nan,
        'future_positives': true_count, 'retrieved_positives': hit_count,
        'groups_with_positive_pct': 100 * groups_positive / len(selected_users),
        'catalog_items': len(items),
    })
    print(snap.split, '· snapshot', snap.snapshot_id,
          '· recall:', round(retrieval_stats[-1]['candidate_recall'], 4),
          '· positive groups:', round(retrieval_stats[-1]['groups_with_positive_pct'], 1), '%')

candidates = pd.concat(tables, ignore_index=True)
candidates.to_parquet(OUT / 'candidates.parquet', index=False)
retrieval_metrics = pd.DataFrame(retrieval_stats)
retrieval_metrics.to_csv(OUT / 'retrieval_metrics.csv', index=False)
display(retrieval_metrics.round(4))

assert all(candidates.groupby(['split', 'group_id'], observed=True).size().eq(N_CANDIDATES))
assert not candidates.duplicated(['split', 'group_id', 'item_id']).any()
assert (snapshots.history_end < snapshots.cutoff).all()


train · 0:   0%|          | 0/10000 [00:00<?, ?it/s]

train · snapshot 0 · recall: 0.2952 · positive groups: 38.8 %


valid · 1:   0%|          | 0/10000 [00:00<?, ?it/s]

valid · snapshot 1 · recall: 0.2974 · positive groups: 40.1 %


test · 2:   0%|          | 0/10000 [00:00<?, ?it/s]

test · snapshot 2 · recall: 0.2919 · positive groups: 39.7 %


,snapshot_id,split,candidate_recall,itemknn_recall,future_positives,retrieved_positives,groups_with_positive_pct,catalog_items
0,0,train,0.2952,0.2744,125061,36912,38.82,625955
1,1,valid,0.2974,0.2759,127294,37858,40.12,627648
2,2,test,0.2919,0.2715,127834,37309,39.70,629298


## Диагностика

In [8]:
print('Число кандидатов на группу:')
display(candidates.groupby(['split', 'group_id']).size().groupby('split').describe().round(2))
print('Средние показатели по split:')
display(retrieval_metrics.groupby('split')[['candidate_recall', 'itemknn_recall',
                                         'groups_with_positive_pct']].mean().round(4))

config = {
    'dataset': REPO, 'source': SOURCE, 'seed': SEED,
    'n_users': N_USERS, 'n_candidates': N_CANDIDATES,
    'n_itemknn': N_ITEMKNN, 'recent_positives': RECENT_POSITIVES,
    'profile_halflife_days': PROFILE_HALFLIFE_DAYS,
    'pop_halflife_days': POP_HALFLIFE_DAYS,
    'test_start': TEST_START, 'valid_start': VALID_START, 'train_start': TRAIN_START,
    'day_seconds': DAY, 'gap_seconds': GAP, 'timestamp_unit': 'seconds rounded to 5',
    'positive_label': 'listen and played_ratio_pct >= 50',
    'candidate_policy': 'binary-cosine ItemKNN on all selected users + decayed popularity',
    'future_positives_added': False,
    'split': 'one train-label day, one valid day, one test day; global GTS with gaps',
    'deviation_from_paper': 'one-day train labels for CatBoost; fixed 600 hybrid candidates',
}
(OUT / 'config.json').write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding='utf-8')
print('Файлы на Google Drive:')
for path in sorted(OUT.iterdir()):
    print(f'{path.name:28} {path.stat().st_size / 1024**2:.1f} MB')


Число кандидатов на группу:


,count,mean,std,min,25%,50%,75%,max
split,,,,,,,,
test,10000.0,600.0,0.0,600.0,600.0,600.0,600.0,600.0
train,10000.0,600.0,0.0,600.0,600.0,600.0,600.0,600.0
valid,10000.0,600.0,0.0,600.0,600.0,600.0,600.0,600.0


Средние показатели по split:


,candidate_recall,itemknn_recall,groups_with_positive_pct
split,,,
test,0.2919,0.2715,39.70
train,0.2952,0.2744,38.82
valid,0.2974,0.2759,40.12


Файлы на Google Drive:
candidates.parquet           178.9 MB
config.json                  0.0 MB
events_sample.parquet        346.1 MB
retrieval_metrics.csv        0.0 MB
selected_users.csv           0.1 MB
snapshots.csv                0.0 MB


## Артефакты